In [1]:
!pip install LiteEFG numpy pandas tqdm

In [2]:
# Write the 3-level IIEFG game file
# ================================================================
# Structure:
#   Level 1 — Chance deals one of 3 cards: Low (L), Mid (M), High (H)
#   Level 2 — Player 1 sees the card: Bet (b) or Check (k)
#   Level 3 — Player 2 sees only P1's action, NOT the card: Call (c) or Fold (f)
#
# Imperfect Information:
#   P2 has 2 infosets — one for all bet nodes, one for all check nodes.
#   P2 cannot distinguish which card was dealt.
# ================================================================

game_content = """# 3-Level IIEFG: Simple Card Game
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 3,
# }
#
node / chance actions L=0.33333333 M=0.33333333 H=0.33333334
node /C:L player 1 actions b k
node /C:M player 1 actions b k
node /C:H player 1 actions b k
node /C:L/P1:b player 2 actions c f
node /C:L/P1:k player 2 actions c f
node /C:M/P1:b player 2 actions c f
node /C:M/P1:k player 2 actions c f
node /C:H/P1:b player 2 actions c f
node /C:H/P1:k player 2 actions c f
node /C:L/P1:b/P2:c leaf payoffs 1=-2 2=2
node /C:L/P1:b/P2:f leaf payoffs 1=1 2=-1
node /C:L/P1:k/P2:c leaf payoffs 1=-1 2=1
node /C:L/P1:k/P2:f leaf payoffs 1=0 2=0
node /C:M/P1:b/P2:c leaf payoffs 1=0 2=0
node /C:M/P1:b/P2:f leaf payoffs 1=1 2=-1
node /C:M/P1:k/P2:c leaf payoffs 1=0 2=0
node /C:M/P1:k/P2:f leaf payoffs 1=0 2=0
node /C:H/P1:b/P2:c leaf payoffs 1=2 2=-2
node /C:H/P1:b/P2:f leaf payoffs 1=1 2=-1
node /C:H/P1:k/P2:c leaf payoffs 1=1 2=-1
node /C:H/P1:k/P2:f leaf payoffs 1=0 2=0
infoset pl1_L nodes /C:L
infoset pl1_M nodes /C:M
infoset pl1_H nodes /C:H
infoset pl2_bet nodes /C:L/P1:b /C:M/P1:b /C:H/P1:b
infoset pl2_check nodes /C:L/P1:k /C:M/P1:k /C:H/P1:k
"""

game_path = "card_game_3level.game"
with open(game_path, "w") as f:
    f.write(game_content)
print(f"Game file written to: {game_path}")

Game file written to: card_game_3level.game


In [3]:
import LiteEFG as efg
from LiteEFG.baselines.CFR import graph as CFR
import pandas as pd
from tqdm import trange

In [4]:
# Load game and register CFR graph
env = efg.FileEnv(game_path, traverse_type="Enumerate")
alg = CFR()
env.set_graph(alg)
print("Game loaded and CFR graph registered.")

===============Graph is ready for CFR===============


Game loaded and CFR graph registered.


In [5]:
# Run CFR — correct training loop from utils.py
NUM_ITER   = 5000
PRINT_FREQ = 500
history    = []
best_exp   = 1e9

for i in trange(NUM_ITER, desc="CFR"):
    alg.update_graph(env)
    # update_best flag accumulates the best-iterate alongside avg-iterate
    env.update_strategy(alg.current_strategy(), update_best=False)

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = sum(env.exploitability(alg.current_strategy(), "avg-iterate"))
        best_exp = min(best_exp, explo)
        history.append({"iteration": i, "exploitability": round(explo, 6), "best": round(best_exp, 6)})

print("\nDone.")

CFR: 100%|██████████████████████████████| 5000/5000 [00:00<00:00, 199201.35it/s]


Done.


In [6]:
# Convergence table
print("--- Exploitability over iterations ---")
print(pd.DataFrame(history).to_string(index=False))

--- Exploitability over iterations ---
 iteration  exploitability     best
         0        0.500000 0.500000
       500        0.000998 0.000998
      1000        0.000500 0.000500
      1500        0.000333 0.000333
      2000        0.000250 0.000250
      2500        0.000200 0.000200
      3000        0.000167 0.000167
      3500        0.000143 0.000143
      4000        0.000125 0.000125
      4500        0.000111 0.000111
      4999        0.000100 0.000100


In [8]:
# Correct signature (from C++ binding error message):
# env.get_strategy(player: int, strategy: GraphNode, type_name: str)
# -> list of (infoset_name, action_probs) tuples

print("=" * 55)
for player_id in range(1, 3):   # 1-indexed
    print(f"\nPlayer {player_id} strategy (avg-iterate):")
    strategy_pairs = env.get_strategy(player_id, alg.current_strategy(), "avg-iterate")
    rows = [{"Infoset": name, "Action Probs [b/k or c/f]": list(probs)}
            for name, probs in strategy_pairs]
    print(pd.DataFrame(rows).to_string(index=False))
print("=" * 55)
print("""
Interpretation:
  P1 (sees card):         actions = [b=Bet, k=Check]
    pl1_H (High card) -> should bet heavily
    pl1_L (Low card)  -> bluff occasionally to keep P2 guessing
    pl1_M (Mid card)  -> mixed strategy

  P2 (blind, sees only P1's action):   actions = [c=Call, f=Fold]
    pl2_bet   -> call if P1's betting range is strong enough
    pl2_check -> call/fold based on expected card distribution
""")


Player 1 strategy (avg-iterate):
Infoset        Action Probs [b/k or c/f]
  pl1_L [0.00010000000000000003, 0.9999]
  pl1_M                       [1.0, 0.0]
  pl1_H                       [1.0, 0.0]

Player 2 strategy (avg-iterate):
  Infoset               Action Probs [b/k or c/f]
  pl2_bet                              [1.0, 0.0]
pl2_check [0.999799999997, 0.0002000000029999994]

Interpretation:
  P1 (sees card):         actions = [b=Bet, k=Check]
    pl1_H (High card) -> should bet heavily
    pl1_L (Low card)  -> bluff occasionally to keep P2 guessing
    pl1_M (Mid card)  -> mixed strategy

  P2 (blind, sees only P1's action):   actions = [c=Call, f=Fold]
    pl2_bet   -> call if P1's betting range is strong enough
    pl2_check -> call/fold based on expected card distribution



In [9]:
# ================================================================
# Interpretation of the Equilibrium (3-Level Imperfect Information Game)
# ================================================================
#
# Game Structure
# --------------
# Chance deals one of three cards with equal probability:
#     L (Low), M (Mid), H (High)
#
# Player 1 observes the card and chooses:
#     b = Bet
#     k = Check
#
# Player 2 observes only Player 1's action (b or k),
# but cannot see the card that was dealt.
#
# Therefore Player 2 has two information sets:
#     pl2_bet   -> all nodes where P1 bet
#     pl2_check -> all nodes where P1 checked
#
# This models imperfect information similar to a simplified poker game.
#
#
# ------------------------------------------------
# Player 1 Strategy Found by CFR
# ------------------------------------------------
#
# pl1_L -> [b ≈ 0, k ≈ 1]
#     With a low card, betting is bad because if P2 calls:
#         payoff = -2
#     Checking leads to:
#         payoff = -1
#     So checking is strictly better.
#
#
# pl1_H -> [b = 1, k = 0]
#     With a high card, betting is best:
#         bet + call  -> +2
#         check + call -> +1
#     Therefore Player 1 always bets with the high card.
#
#
# pl1_M -> [b = 1, k = 0]  (solver chose pure bet)
#     With the mid card:
#         bet + call  -> 0
#         check + call -> 0
#     Both actions give the same payoff, so Player 1 is indifferent.
#     CFR therefore converges to one valid equilibrium (betting).
#
#
# ------------------------------------------------
# Player 2 Strategy Found by CFR
# ------------------------------------------------
#
# pl2_bet -> [c = 1, f = 0]
#     Player 2 always calls when Player 1 bets.
#
#     If Player 2 folds:
#         Player 1 gets +1 for sure.
#
#     If Player 2 calls:
#         outcomes depend on the card,
#         but overall calling is at least as good as folding.
#
#
# pl2_check -> [c ≈ 1, f ≈ 0]
#     Player 2 almost always calls after a check.
#
#     Because strong hands tend to bet,
#     the check range contains many weaker hands.
#     Calling therefore performs well.
#
#
# ------------------------------------------------
# Why There Is No Bluffing in This Game
# ------------------------------------------------
#
# In many poker-like games (e.g., Kuhn Poker),
# Player 1 bluffs with weak hands occasionally.
#
# Bluffing only occurs if:
#     Player 2 sometimes folds to bets.
#
# In this payoff structure:
#     Player 2's best response is to call almost always.
#
# Since Player 2 rarely folds,
# bluffing with the low card becomes unprofitable.
#
# Therefore the equilibrium becomes:
#
#     Low card  -> check
#     Mid card  -> bet (indifferent)
#     High card -> bet
#
# and Player 2 calls almost everything.
#
#
# ------------------------------------------------
# Key Insight About Imperfect Information Games
# ------------------------------------------------
#
# Strategic mixing (bluffing) appears only when:
#
#     1. Players are uncertain about hidden information
#     2. Folding sometimes becomes optimal
#
# If calling is always best, bluffing disappears.
#
# This is why more carefully balanced payoff structures
# (such as Kuhn Poker) produce mixed strategies.
#
# ================================================================